# 09 — XAI Business Roll-Up (Group → Sub_Group → Feature)

Rolls up every feature-level XAI importance file to the two higher hierarchy levels
defined in `feature_hierarchy.csv`.

| Level | Description |
|-------|-------------|
| `Feature` | Original model feature (e.g. `lag_1`, `is_promo`) |
| `Sub_Group` | Derived factor (e.g. `Recent Lags`, `Promo State`) |
| `Group` | Business group (e.g. `Demand History`, `Promotional Activity`) |

**Weighting rule:** Equal weights within each level, renormalized over features  
present in each model's output. Models are rolled up independently.

**Outputs** (never overwrites originals):
- `Outputs/shap_importance_rolled.csv`
- `Outputs/lime_importance_rolled.csv`
- `Outputs/surrogate_importance_rolled.csv`

In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
from src.rollup_utils import load_hierarchy, rollup, validate_weights

NOTEBOOK_DIR = os.path.abspath('')
OUTPUT_DIR   = os.path.join(NOTEBOOK_DIR, '..', 'Outputs')
HIERARCHY    = os.path.join(NOTEBOOK_DIR, '..', 'feature_hierarchy.csv')

hierarchy = load_hierarchy(HIERARCHY)
print(f'Hierarchy loaded: {len(hierarchy)} features across {hierarchy["Group"].nunique()} groups')
print(hierarchy.groupby('Group')['Sub_Group'].nunique().rename('n_sub_groups').to_string())

## 1. SHAP Roll-Up

In [ ]:
shap_path = os.path.join(OUTPUT_DIR, 'shap_importance.csv')
shap_df   = pd.read_csv(shap_path)
print(f'SHAP input  : {len(shap_df):,} rows  |  models: {sorted(shap_df["Model"].unique())}')

shap_rolled = rollup(shap_df, hierarchy, value_col='Mean_Abs_SHAP')
validate_weights(shap_rolled, 'Mean_Abs_SHAP')

out = os.path.join(OUTPUT_DIR, 'shap_importance_rolled.csv')
shap_rolled.to_csv(out, index=False)
print(f'SHAP output : {len(shap_rolled):,} rows → {os.path.basename(out)}')
print(shap_rolled['Level'].value_counts().to_string())

## 2. LIME Roll-Up

In [ ]:
lime_path = os.path.join(OUTPUT_DIR, 'lime_importance.csv')
lime_df   = pd.read_csv(lime_path)
print(f'LIME input  : {len(lime_df):,} rows  |  models: {sorted(lime_df["Model"].unique())}')

lime_rolled = rollup(lime_df, hierarchy, value_col='Mean_Abs_LIME')
validate_weights(lime_rolled, 'Mean_Abs_LIME')

out = os.path.join(OUTPUT_DIR, 'lime_importance_rolled.csv')
lime_rolled.to_csv(out, index=False)
print(f'LIME output : {len(lime_rolled):,} rows → {os.path.basename(out)}')
print(lime_rolled['Level'].value_counts().to_string())

## 3. Surrogate Roll-Up

`Surrogate_Model` (e.g. GLS, DT, LR, RE_Panel) is included as an extra slice column  
so each surrogate type rolls up independently.

In [ ]:
surr_path = os.path.join(OUTPUT_DIR, 'surrogate_importance.csv')
surr_df   = pd.read_csv(surr_path)
print(f'Surrogate input : {len(surr_df):,} rows  |  surrogate models: {sorted(surr_df["Surrogate_Model"].unique())}')

surr_rolled = rollup(surr_df, hierarchy, value_col='Feature_Importance',
                     extra_group_cols=['Surrogate_Model'])
validate_weights(surr_rolled, 'Feature_Importance', extra_group_cols=['Surrogate_Model'])

out = os.path.join(OUTPUT_DIR, 'surrogate_importance_rolled.csv')
surr_rolled.to_csv(out, index=False)
print(f'Surrogate output: {len(surr_rolled):,} rows → {os.path.basename(out)}')
print(surr_rolled['Level'].value_counts().to_string())

## 4. Quick Views

In [ ]:
# Group-level comparison across models — DEV Global
for label, df_r, vcol in [
    ('SHAP',      shap_rolled, 'Mean_Abs_SHAP'),
    ('LIME',      lime_rolled, 'Mean_Abs_LIME'),
    ('Surrogate', surr_rolled, 'Feature_Importance'),
]:
    subset = df_r[
        (df_r['Level'] == 'Group') &
        (df_r['Dataset'] == 'DEV') &
        (df_r['Group_Variable'] == 'Global')
    ].copy()
    if 'Surrogate_Model' in subset.columns:
        subset = subset[subset['Surrogate_Model'] == subset['Surrogate_Model'].iloc[0]]
    pivot = subset.pivot_table(index='Group', columns='Model', values=vcol).round(4)
    print(f'\n── {label} — Group Level, DEV Global ──────────────────────────')
    display(pivot)